# Project 4 — GPW Stock Portfolio Optimization (Markowitz Model)
## Enterprise Risk Management

**Context.** **XTB S.A.** received **PLN 1 million** to invest exclusively in stocks
listed on the **Warsaw Stock Exchange (GPW)**. Our task is to select a portfolio, compute risk measures,
and compare the results with forecasts from a single-index model (CAPM-like).

---

## Selection of 5 Polish GPW Stocks and Rationale

Selection criteria: **sector diversification**, **high liquidity** (WIG20 blue chips),
**stability** (no delisting risk), and **representativeness of the Polish economy**.

| # | Ticker | Company | Sector | Rationale |
|---|--------|--------|--------|--------------|
| 1 | **PKO.WA** | PKO Bank Polski | Banks | Largest Polish bank, high dividends, β ≈ market, exposure to NBP policy rate |
| 2 | **PKN.WA** | PKN Orlen | Energy / fuels | State-owned giant, hedge against commodity inflation, high turnover |
| 3 | **KGH.WA** | KGHM Polish Copper | Commodities (copper, silver) | Global USD/CNY exposure, decorrelation from banks, high volatility |
| 4 | **CDR.WA** | CD Projekt | Technology / gaming | IT sector, global revenues in EUR/USD, low correlation with financials |
| 5 | **DNP.WA** | Dino Polska | Consumer staples (food retail) | Defensive, low β, stable cash flow — balance for the portfolio |

**Why this selection?** We obtain exposure to **5 independent sectors**
(banks, energy, commodities, IT, retail), which maximizes diversification benefits.
All companies are in **WIG20** — the highest liquidity on GPW, allowing easy opening and closing
of positions for PLN 1 million without significant *slippage*.

**Risk-free rate.** We assume the yield on Polish short-term government bonds
(52-week bills) at **r_f ≈ 5.25% per annum** (approximate level
after the NBP rate-cut cycle in 2025).

**Market index.** **WIG20** — benchmark of large GPW stocks (ticker `WIG20.WA` on Yahoo Finance).

In Section 1 (two-asset portfolio) we use **PKO.WA + CDR.WA** — a banks + tech combination
that empirically shows **low correlation** and well illustrates the diversification effect.

In [ ]:
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import stats
from scipy.stats import norm
from scipy.optimize import minimize
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.size'] = 11

# --- Project parameters ---
TICKERS_5 = ['PKO.WA', 'PKN.WA', 'KGH.WA', 'CDR.WA', 'DNP.WA',
             'VOT.WA', 'KRU.WA', 'PZU.WA', 'ENT.WA', 'DVL.WA',
             'DOM.WA', 'ALE.WA', 'ACP.WA']
NAMES = {
    'PKO.WA': 'PKO BP',
    'PKN.WA': 'PKN Orlen',
    'KGH.WA': 'KGHM',
    'CDR.WA': 'CD Projekt',
    'DNP.WA': 'Dino Polska',
    'VOT.WA': 'VOTUM',
    'KRU.WA': 'Kruk',
    'PZU.WA': 'PZU',
    'ENT.WA': 'Enter Air',
    'DVL.WA': 'Develia',
    'DOM.WA': 'Dom Development',
    'ALE.WA': 'Allegro',
    'ACP.WA': 'Asseco',
}
INDEX_NAME = 'WIG20'
INDEX_CSV  = 'wig20_d.csv'  # load WIG20 from local file

START   = '2021-01-02'
END     = '2026-05-13'
RF_ANN  = 0.05           # 5.25% per annum (short-term Polish bonds)
TRADING_DAYS = 252
RF_D    = RF_ANN / TRADING_DAYS

CAPITAL_PLN = 1_000_000    # PLN 1 million to invest

print(f'Stocks ({len(TICKERS_5)}): {", ".join(NAMES.values())}')
print(f'Index: {INDEX_NAME} (file {INDEX_CSV})')
print(f'Period: {START} — {END}')
print(f'r_f annual: {RF_ANN:.2%}   r_f daily: {RF_D:.6f}')


In [ ]:
# --- Stocks from Yahoo Finance, WIG20 from local CSV ---
def _extract_close(raw, tickers):
    """Extract 'Adj Close' (or 'Close') as DataFrame[tickers]."""
    if isinstance(raw.columns, pd.MultiIndex):
        lvl0 = raw.columns.get_level_values(0)
        field = 'Adj Close' if 'Adj Close' in lvl0 else 'Close'
        df = raw[field].copy()
    else:
        field = 'Adj Close' if 'Adj Close' in raw.columns else 'Close'
        df = raw[[field]].copy()
        df.columns = tickers
    return df[tickers]

# 1) Stocks from Yahoo
stocks_raw = yf.download(TICKERS_5, start=START, end=END,
                         progress=False, auto_adjust=False)
stocks = _extract_close(stocks_raw, TICKERS_5).dropna(how='all')
stocks.columns = [NAMES[c] for c in stocks.columns]

# 2) WIG20 from local CSV file (columns: Data, Otwarcie, Najwyzszy, Najnizszy, Zamkniecie, Wolumen)
wig20_df = pd.read_csv(INDEX_CSV)
wig20_df['Data'] = pd.to_datetime(wig20_df['Data'])
wig20 = (wig20_df.set_index('Data').sort_index()
                 .loc[START:END, 'Zamkniecie'])
wig20.name = INDEX_NAME

# 3) Merge and clean
prices = stocks.join(wig20.to_frame(), how='inner').dropna()

# log-returns
log_ret = np.log(prices / prices.shift(1)).dropna()

ret_index = log_ret[INDEX_NAME]
log_ret_assets = log_ret[[NAMES[t] for t in TICKERS_5]]

assert len(log_ret_assets) > 0, 'No overlapping observations for stocks and index.'

print(f'Observations: {len(log_ret_assets)} trading days')
print(f'Range:       {log_ret_assets.index[0].date()} — {log_ret_assets.index[-1].date()}')
display(prices.tail())

In [ ]:
# --- Visualization: prices and log-returns ---
fig, axes = plt.subplots(2, 1, figsize=(15, 9))

# Prices normalized to 100
norm_prices = prices[[NAMES[t] for t in TICKERS_5]] / prices[[NAMES[t] for t in TICKERS_5]].iloc[0] * 100
norm_prices.plot(ax=axes[0], lw=1.2)
axes[0].set_title('Prices normalized to 100 (period start)', fontweight='bold')
axes[0].set_ylabel('Index (start = 100)')
axes[0].grid(alpha=0.3); axes[0].legend(loc='upper left')

# Log-returns
log_ret_assets.plot(ax=axes[1], lw=0.5, alpha=0.7)
axes[1].set_title('Logarithmic returns (daily)', fontweight='bold')
axes[1].set_ylabel('log-return')
axes[1].axhline(0, color='black', lw=0.5); axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

---
# 1. **Two-asset** portfolio — PKO BP & CD Projekt

We select a pair from different sectors (banks vs tech) to demonstrate the diversification effect.

### Notation
- $R_p = w_1 R_1 + w_2 R_2,\quad w_1 + w_2 = 1$
- $\mathbb{E}(R_p) = w_1 \mu_1 + w_2 \mu_2$
- $\sigma_p^2 = w_1^2 \sigma_1^2 + w_2^2 \sigma_2^2 + 2 w_1 w_2 \rho_{12}\sigma_1\sigma_2$

In [ ]:
# --- 1. Statistics for PKO BP & CD Projekt pair ---
pair = ['PKO BP', 'PKN Orlen']
R = log_ret_assets[pair]

mu  = R.mean() * TRADING_DAYS          # annual
sig = R.std()  * np.sqrt(TRADING_DAYS) # annual
rho = R.corr().iloc[0, 1]

print('--- Annual statistics ---')
print(f'  E(R_{pair[0]}) = {mu.iloc[0]:.4f}   σ = {sig.iloc[0]:.4f}')
print(f'  E(R_{pair[1]}) = {mu.iloc[1]:.4f}   σ = {sig.iloc[1]:.4f}')
print(f'  Correlation ρ_{{12}} = {rho:.4f}')

Sigma2 = R.cov().values * TRADING_DAYS  # annual covariance matrix
mu_v   = mu.values
print('\nCovariance matrix (annual):')
display(pd.DataFrame(Sigma2, index=pair, columns=pair).round(6))

## 1a + 1b — Portfolio return and risk for different weights

We plot the set of attainable portfolios $(\sigma_p, \mathbb{E}(R_p))$.
We also mark two specific portfolios:
- **equal weights** $w = (0.5,\,0.5)$
- **market-cap weights** (approximate — see cell).

In [ ]:
# --- 1a/1b: attainable portfolio curve ---
w_grid = np.linspace(0, 1, 201)
mu_p  = np.array([w*mu_v[0] + (1-w)*mu_v[1] for w in w_grid])
sig_p = np.array([np.sqrt(w**2*Sigma2[0,0] + (1-w)**2*Sigma2[1,1]
                          + 2*w*(1-w)*Sigma2[0,1]) for w in w_grid])

# Equal-weight portfolio
w_eq = np.array([0.5, 0.5])
mu_eq  = w_eq @ mu_v
sig_eq = np.sqrt(w_eq @ Sigma2 @ w_eq)

# Market-cap weights (indicative — May 2025)
#   PKO BP  ≈  75 mld PLN
#   CD PROJEKT ≈ 16 mld PLN
cap = np.array([75.0, 16.0])
w_cap = cap / cap.sum()
mu_cap  = w_cap @ mu_v
sig_cap = np.sqrt(w_cap @ Sigma2 @ w_cap)

print(f'Equal-weight portfolio (50/50):        E(R)={mu_eq:.4f}, σ={sig_eq:.4f}')
print(f'Market-cap-weighted portfolio:         w={w_cap.round(3)}  →  E(R)={mu_cap:.4f}, σ={sig_cap:.4f}')

## 1c — **Minimum-variance** portfolio (MVP)

For two assets (no short selling and $w_1+w_2=1$):
$$
w_1^{*} = \frac{\sigma_2^2 - \rho\,\sigma_1\sigma_2}{\sigma_1^2 + \sigma_2^2 - 2\rho\,\sigma_1\sigma_2}
$$

In [ ]:
# --- 1c: MVP analytically ---
s1, s2 = sig.iloc[0], sig.iloc[1]
w1_mvp = (s2**2 - rho*s1*s2) / (s1**2 + s2**2 - 2*rho*s1*s2)
w1_mvp = float(np.clip(w1_mvp, 0, 1))
w_mvp  = np.array([w1_mvp, 1 - w1_mvp])
mu_mvp  = w_mvp @ mu_v
sig_mvp = np.sqrt(w_mvp @ Sigma2 @ w_mvp)
print(f'MVP:  w = ({w_mvp[0]:.4f}, {w_mvp[1]:.4f})  →  E(R)={mu_mvp:.4f},  σ={sig_mvp:.4f}')

## 1d — Minimum-risk portfolio at a **target return** $R^\star$

For a two-asset pair, $R^\star$ uniquely determines the weights:
$w_1 = \frac{R^\star - \mu_2}{\mu_1 - \mu_2}$, $w_2 = 1 - w_1$.
We set $R^\star = 0{,}15$ (15% per annum).

In [ ]:
R_target = 0.15
w1_t = (R_target - mu_v[1]) / (mu_v[0] - mu_v[1])
w1_t = float(np.clip(w1_t, 0, 1))
w_target = np.array([w1_t, 1 - w1_t])
mu_t  = w_target @ mu_v
sig_t = np.sqrt(w_target @ Sigma2 @ w_target)
print(f'Target R*=0.15: w=({w_target[0]:.4f}, {w_target[1]:.4f})  →  E(R)={mu_t:.4f},  σ={sig_t:.4f}')

## 1e — **Market** portfolio (max Sharpe)

The capital market line (CML) is tangent to the efficient frontier at the market portfolio.
We maximize $S(w) = \dfrac{\mathbb{E}(R_p) - r_f}{\sigma_p}$.

In [ ]:
# --- 1e: market portfolio (max Sharpe) ---
def sharpe(w, mu, Sigma, rf=RF_ANN):
    rp  = w @ mu
    sp  = np.sqrt(w @ Sigma @ w)
    return (rp - rf) / sp

S_grid = np.array([sharpe(np.array([w, 1-w]), mu_v, Sigma2) for w in w_grid])
i_max  = int(np.argmax(S_grid))
w_mkt  = np.array([w_grid[i_max], 1 - w_grid[i_max]])
mu_mkt  = w_mkt @ mu_v
sig_mkt = np.sqrt(w_mkt @ Sigma2 @ w_mkt)
S_mkt   = (mu_mkt - RF_ANN) / sig_mkt
print(f'Market portfolio: w=({w_mkt[0]:.4f}, {w_mkt[1]:.4f})')
print(f'                  E(R)={mu_mkt:.4f},  σ={sig_mkt:.4f},  Sharpe={S_mkt:.4f}')

In [ ]:
# --- Plot: attainable portfolios + MVP + market portfolio + CML ---
fig, ax = plt.subplots(figsize=(11, 7))

ax.plot(sig_p, mu_p, color='steelblue', lw=2, label='Set of attainable portfolios')
ax.scatter(sig.iloc[0], mu_v[0], s=120, marker='^', color='navy',
           label=f'100% {pair[0]}', zorder=4)
ax.scatter(sig.iloc[1], mu_v[1], s=120, marker='v', color='darkorange',
           label=f'100% {pair[1]}', zorder=4)
ax.scatter(sig_eq, mu_eq,  s=110, color='gray', label='50/50', zorder=4)
ax.scatter(sig_cap, mu_cap, s=110, color='purple', marker='D', label='Weights ~ mkt cap', zorder=4)
ax.scatter(sig_mvp, mu_mvp, s=160, color='green', marker='*',
           label=f'MVP (w={w_mvp[0]:.2f})', zorder=5)
ax.scatter(sig_t, mu_t,    s=130, color='blue',  marker='P',
           label=f'R*=0.15 (w={w_target[0]:.2f})', zorder=5)
ax.scatter(sig_mkt, mu_mkt, s=180, color='red', marker='*',
           label=f'Market portfolio (w={w_mkt[0]:.2f})', zorder=5)

# Linia CML
x_cml = np.linspace(0, sig.max()*1.1, 100)
y_cml = RF_ANN + (mu_mkt - RF_ANN)/sig_mkt * x_cml
ax.plot(x_cml, y_cml, '--', color='red', alpha=0.6, label='CML')

ax.axhline(RF_ANN, color='black', ls=':', alpha=0.5, label=f'r_f={RF_ANN:.2%}')
ax.set_xlabel('Risk σ_p (annual)'); ax.set_ylabel('Expected return E(R_p)')
ax.set_title('Sec. 1: Two-asset attainable portfolios (PKO BP + CD Projekt)',
             fontweight='bold')
ax.legend(loc='best'); ax.grid(alpha=0.3); ax.set_xlim(left=0)
plt.tight_layout(); plt.show()

---
# 2. Portfolio VaR from Section 1 using variance-covariance method

For a portfolio with return $R_p \sim \mathcal{N}(\mu_p, \sigma_p^2)$ (method assumption):
$$
\mathrm{VaR}_{\alpha}^{T} = -\big(\mu_p \cdot T + z_\alpha \cdot \sigma_p\cdot\sqrt{T}\big)
$$
where $z_\alpha = \Phi^{-1}(\alpha)$, $\alpha\in\{0{,}05,\,0{,}01\}$.

### Additional constraint
> *With 99% probability, the portfolio cannot lose more than 20% of its value over one year.*

That is: **$\mathrm{VaR}_{99\%}^{1\,\text{year}} \le 0{,}20$**, i.e.
$\mu_p - 2{,}3263\,\sigma_p \ge -0{,}20$.

In [ ]:
# --- 2. VaR (variance-covariance) for Section 1 portfolios ---
def var_param(mu, sig, alpha, T=1):
    # annual mu, sig; T in years
    z = norm.ppf(alpha)
    return -(mu*T + z*sig*np.sqrt(T))

def es_param(mu, sig, alpha, T=1):
    # Expected Shortfall under normality
    return -(mu*T - sig*np.sqrt(T) * norm.pdf(norm.ppf(alpha))/alpha)

def check_constraint_20pct(mu, sig):
    # Does VaR 99% (annual) <= 20%?
    return var_param(mu, sig, 0.01, T=1) <= 0.20

ports = {
    'PKO BP (100%)':       (mu_v[0], sig.iloc[0]),
    'PKN Orlen (100%)':   (mu_v[1], sig.iloc[1]),
    'Equal-weight 50/50':   (mu_eq,   sig_eq),
    'Mkt-cap weighted':     (mu_cap,  sig_cap),
    'MVP':                 (mu_mvp,  sig_mvp),
    f'R*=0.15':            (mu_t,    sig_t),
    'Market (max Sharpe)':(mu_mkt,  sig_mkt),
}

rows = []
for name, (m, s) in ports.items():
    rows.append({
        'Portfolio':       name,
        'E(R) annual':  f'{m:.4f}',
        'σ annual':     f'{s:.4f}',
        'VaR 95% (yr)': f'{var_param(m, s, 0.05):.4f}',
        'VaR 99% (yr)': f'{var_param(m, s, 0.01):.4f}',
        'ES  99% (yr)': f'{es_param(m, s, 0.01):.4f}',
        'Meets ≤20%?': 'YES ✓' if check_constraint_20pct(m, s) else 'NO ✗',
    })
var_table = pd.DataFrame(rows)
display(var_table)

## Portfolio maximizing $E(R)$ subject to VaR 99% ≤ 20%

From the linear constraint $\mu_p - z_{0.01}\sigma_p \ge -0{,}20$ we determine
the boundary of feasible weights — along the set of attainable portfolios we search for the maximum
$E(R_p)$.

In [ ]:
# --- 2. Optimization: max E(R) | VaR_99% <= 0.20 ---
z99 = -norm.ppf(0.01)   # ≈ 2.3263

feas_mask = (mu_p - z99 * sig_p) >= -0.20
if feas_mask.any():
    mu_p_feas = mu_p.copy(); mu_p_feas[~feas_mask] = -np.inf
    i_best = int(np.argmax(mu_p_feas))
    w_best = np.array([w_grid[i_best], 1 - w_grid[i_best]])
    mu_b, sig_b = mu_p[i_best], sig_p[i_best]
    print(f'Maximum E(R) subject to VaR99% ≤ 20%:')
    print(f'   w = ({w_best[0]:.4f}, {w_best[1]:.4f})')
    print(f'   E(R) = {mu_b:.4f},  σ = {sig_b:.4f}')
    print(f'   VaR 99% (rok) = {var_param(mu_b, sig_b, 0.01):.4f}')
else:
    print('No portfolio from the pair satisfies the ≤ 20% loss constraint at 99% probability.')

# Visualization of feasible region
fig, ax = plt.subplots(figsize=(11, 6))
ax.plot(sig_p, mu_p, color='lightgray', lw=2, label='Set of attainable portfolios')
ax.plot(sig_p[feas_mask], mu_p[feas_mask], color='green', lw=3,
        label='Meets VaR99% ≤ 20%')
ax.scatter(sig_mvp, mu_mvp, s=140, color='blue', marker='*', label='MVP')
ax.scatter(sig_mkt, mu_mkt, s=140, color='red', marker='*', label='Market portfolio')
if feas_mask.any():
    ax.scatter(sig_b, mu_b, s=200, color='orange', marker='P',
               label=f'Max E(R) | VaR≤20% (w={w_best[0]:.2f})', zorder=5)
# constraint line: mu = z99*sig - 0.20
x_c = np.linspace(0, sig_p.max()*1.1, 100)
ax.plot(x_c, z99*x_c - 0.20, '--', color='red', alpha=0.6, label='VaR99%=20% boundary')
ax.set_xlabel('σ_p'); ax.set_ylabel('E(R_p)')
ax.set_title('Sec. 2: Feasible portfolios (VaR 99% ≤ 20% annual loss)',
             fontweight='bold')
ax.legend(); ax.grid(alpha=0.3); ax.set_xlim(left=0)
plt.tight_layout(); plt.show()

## Sec. 2 (extension) — adding bonds (risk-free asset)

We add a third component to the portfolio — **government bonds** at the reference rate
$r_f =$ `RF_ANN` (assumption: $\sigma_B = 0$, $\rho_{B,\cdot}=0$).

Let $f_B$ be the bond share of the total portfolio. Then for a two-asset portfolio
(PKO BP + CD Projekt) with weights $(w, 1-w)$ and parameters $(\mu_p, \sigma_p)$ we have:
$$
\mathbb{E}(R) = f_B\,r_f + (1-f_B)\,\mu_p,\qquad
\sigma = (1-f_B)\,\sigma_p.
$$

The attainable portfolio curve **scales linearly** toward the point $(\sigma=0,\,\mu=r_f)$
as $f_B$ increases. We plot 5 charts: $f_B \in \{0\%, 20\%, 40\%, 60\%, 80\%\}$.

In [ ]:
# --- 2 (extension): adding risk-free bonds (r_f = RF_ANN) ---
# For bond share f_B:  mu = f_B*r_f + (1-f_B)*mu_p,   sig = (1-f_B)*sig_p
bond_fractions = [0.0, 0.20, 0.40, 0.60, 0.80]

# VaR99% = 20% boundary:  mu = z99*sig - 0.20
z99 = -norm.ppf(0.01)

# Shared axes for all subplots (for comparability)
xmax = sig_p.max() * 1.10
ymin = min(mu_p.min(), 0) - 0.05
ymax = mu_p.max() + 0.05

fig, axes = plt.subplots(2, 3, figsize=(17, 10))
axes = axes.flatten()

summary_rows = []

for ax, f_B in zip(axes[:5], bond_fractions):
    # Combined portfolios (stocks + bonds)
    mu_c  = f_B * RF_ANN + (1 - f_B) * mu_p
    sig_c = (1 - f_B) * sig_p

    # Feasibility mask: VaR99% <= 20%
    feas = (mu_c - z99 * sig_c) >= -0.20

    # Curve of all attainable portfolios (stocks, no bonds) — background
    ax.plot(sig_p, mu_p, color='lightgray', lw=1.5, ls='--',
            label='No bonds (reference)')

    # Portfolio curve with f_B bonds
    ax.plot(sig_c, mu_c, color='steelblue', lw=2,
            label=f'Stocks + {f_B*100:.0f}% bonds')
    if feas.any():
        ax.plot(sig_c[feas], mu_c[feas], color='green', lw=3.5,
                label='Meets VaR99% ≤ 20%')

    # Bond point (when f_B>0 — illustrative r_f point only)
    ax.scatter(0, RF_ANN, s=110, marker='s', color='black',
               label=f'Bonds (r_f={RF_ANN:.2%})', zorder=5)

    # Maximum E(R) subject to constraint
    if feas.any():
        mu_c_feas = mu_c.copy(); mu_c_feas[~feas] = -np.inf
        i_best = int(np.argmax(mu_c_feas))
        w_stock = w_grid[i_best]
        mu_b, sig_b = mu_c[i_best], sig_c[i_best]
        ax.scatter(sig_b, mu_b, s=200, color='orange', marker='P',
                   edgecolor='black',
                   label=f'Max E(R)|VaR≤20%\nw_PKO={w_stock*(1-f_B):.2f}, '
                         f'w_PKN={(1-w_stock)*(1-f_B):.2f}', zorder=6)
        summary_rows.append({
            'f_B (bonds)': f'{f_B*100:.0f}%',
            'w_PKO': round(w_stock*(1-f_B), 4),
            'w_PKN': round((1-w_stock)*(1-f_B), 4),
            'w_Bond': round(f_B, 4),
            'E(R)':   round(mu_b, 4),
            'σ':      round(sig_b, 4),
            'VaR 99% (yr)': round(var_param(mu_b, sig_b, 0.01), 4),
        })
    else:
        summary_rows.append({
            'f_B (bonds)': f'{f_B*100:.0f}%',
            'w_PKO': None, 'w_PKN': None, 'w_Bond': round(f_B, 4),
            'E(R)': None, 'σ': None, 'VaR 99% (rok)': None,
        })

    # VaR99% = 20% boundary
    x_c = np.linspace(0, xmax, 100)
    ax.plot(x_c, z99 * x_c - 0.20, '--', color='red', alpha=0.6,
            label='VaR99%=20% boundary')

    # r_f as horizontal reference
    ax.axhline(RF_ANN, color='black', ls=':', alpha=0.4)

    ax.set_xlim(-0.005, xmax)
    ax.set_ylim(ymin, ymax)
    ax.set_xlabel('σ_p')
    ax.set_ylabel('E(R_p)')
    ax.set_title(f'Bonds = {f_B*100:.0f}%   (stocks = {(1-f_B)*100:.0f}%)',
                 fontweight='bold')
    ax.grid(alpha=0.3)
    ax.legend(loc='lower right', fontsize=8)

# 6th panel: aggregate comparison
ax6 = axes[5]
cmap = plt.cm.viridis(np.linspace(0, 0.85, len(bond_fractions)))
for f_B, col in zip(bond_fractions, cmap):
    mu_c  = f_B * RF_ANN + (1 - f_B) * mu_p
    sig_c = (1 - f_B) * sig_p
    ax6.plot(sig_c, mu_c, color=col, lw=2, label=f'{f_B*100:.0f}% bonds')
ax6.scatter(0, RF_ANN, s=90, marker='s', color='black', zorder=5)
x_c = np.linspace(0, xmax, 100)
ax6.plot(x_c, z99 * x_c - 0.20, '--', color='red', alpha=0.6,
         label='VaR99%=20% boundary')
ax6.axhline(RF_ANN, color='black', ls=':', alpha=0.4, label=f'r_f={RF_ANN:.2%}')
ax6.set_xlim(-0.005, xmax); ax6.set_ylim(ymin, ymax)
ax6.set_xlabel('σ_p'); ax6.set_ylabel('E(R_p)')
ax6.set_title('Comparison: all bond allocations', fontweight='bold')
ax6.grid(alpha=0.3); ax6.legend(loc='lower right', fontsize=9)

plt.suptitle('Sec. 2 (extension): Two-stock portfolio + bonds (r_f = RF_ANN)',
             fontweight='bold', fontsize=13, y=1.00)
plt.tight_layout()
plt.show()

print('\nMax E(R) subject to VaR99% ≤ 20% — for different bond allocations:')
display(pd.DataFrame(summary_rows))
# --- 3: Plot E(R_p) vs VaR99% for different bond allocations ---
fig, ax = plt.subplots(figsize=(10, 7))

colors = plt.cm.viridis(np.linspace(0, 0.9, len(bond_fractions)))

for f_B, c in zip(bond_fractions, colors):
    mu_c  = f_B * RF_ANN + (1 - f_B) * mu_p
    sig_c = (1 - f_B) * sig_p
    VaR_c = z99 * sig_c - mu_c            # VaR99% (loss as positive number)

    feas = VaR_c <= 0.20

    ax.plot(VaR_c, mu_c, color=c, lw=2,
            label=f'Stocks + {f_B*100:.0f}% bonds')

    # Maximum E(R) subject to constraint VaR <= 20%
    if feas.any():
        mu_c_feas = np.where(feas, mu_c, -np.inf)
        i_best = int(np.argmax(mu_c_feas))
        ax.scatter(VaR_c[i_best], mu_c[i_best], s=160, marker='P',
                color=c, edgecolor='black', zorder=5)

# VaR = 20% boundary
ax.axvline(0.20, color='red', ls='--', lw=1.5, label='VaR99% = 20%')
ax.axvspan(0.20, ax.get_xlim()[1] if ax.get_xlim()[1] > 0.20 else 1.0,
             color='red', alpha=0.07)

# Bond point (VaR ≈ -r_f, because no volatility)
ax.scatter(-RF_ANN, RF_ANN, s=120, marker='s', color='black',
             label=f'Bonds (r_f={RF_ANN:.2%})', zorder=6)

ax.axhline(0, color='gray', lw=0.6)
ax.axvline(0, color='gray', lw=0.6)
ax.set_xlabel('VaR99% (annual loss)')
ax.set_ylabel('E(R_p) — expected return')
ax.set_title('E(R) – VaR99% profile for stock+bond portfolios')
ax.legend(loc='best', fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


---
# 3. Portfolio extension to **5 stocks** (numerical optimization)

Vector notation:
$$
\mathbb{E}(R_p) = \mathbf{w}^\top \boldsymbol{\mu},\qquad
\sigma_p^2 = \mathbf{w}^\top \Sigma\, \mathbf{w},\qquad
\sum_i w_i = 1,\ \ w_i \ge 0\ (\text{no shorting).}
$$

We compute:
- **efficient frontier** (Markowitz) — by sweeping over target return,
- **MVP**, **market portfolio (max Sharpe)** — `scipy.optimize.minimize`,
- **Monte Carlo cloud** (10,000 random portfolios) — background.

In [ ]:
# --- 3. Five-stock portfolio: vector data ---
R5      = log_ret_assets[[NAMES[t] for t in TICKERS_5]]
mu5     = R5.mean().values * TRADING_DAYS
Sigma5  = R5.cov().values * TRADING_DAYS
names5  = list(R5.columns)
n = len(names5)

print('Annual expected returns E(R):')
display(pd.Series(mu5, index=names5, name='E(R)').round(4))

print('\nCorrelation matrix:')
display(R5.corr().round(3))

print('\nCovariance matrix (annual):')
display(pd.DataFrame(Sigma5, index=names5, columns=names5).round(5))

In [ ]:
# --- 3. Monte Carlo: cloud of 10,000 random portfolios ---
np.random.seed(42)
N_MC = 10_000
W = np.random.dirichlet(np.ones(n), size=N_MC)   # weights summing to 1, w_i>=0
mc_mu  = W @ mu5
mc_sig = np.sqrt(np.einsum('ij,jk,ik->i', W, Sigma5, W))
mc_sharpe = (mc_mu - RF_ANN) / mc_sig

In [ ]:
# --- 3. MVP, market portfolio, efficient frontier — numerical optimization ---
def port_var(w, S): return w @ S @ w
def port_ret(w, m): return w @ m
def neg_sharpe(w, m, S, rf): return -(w @ m - rf) / np.sqrt(w @ S @ w)

bounds = tuple((0.0, 1.0) for _ in range(n))
cons_sum = {'type':'eq', 'fun': lambda w: w.sum() - 1.0}

# MVP
res_mvp = minimize(port_var, x0=np.ones(n)/n, args=(Sigma5,), method='SLSQP',
                   bounds=bounds, constraints=[cons_sum])
w_mvp5  = res_mvp.x
mu_mvp5  = port_ret(w_mvp5, mu5)
sig_mvp5 = np.sqrt(port_var(w_mvp5, Sigma5))

# Market portfolio
res_mkt = minimize(neg_sharpe, x0=np.ones(n)/n, args=(mu5, Sigma5, RF_ANN),
                   method='SLSQP', bounds=bounds, constraints=[cons_sum])
w_mkt5  = res_mkt.x
mu_mkt5  = port_ret(w_mkt5, mu5)
sig_mkt5 = np.sqrt(port_var(w_mkt5, Sigma5))
S_mkt5   = (mu_mkt5 - RF_ANN) / sig_mkt5

# Efficient frontier: sweep
mu_targets = np.linspace(mu_mvp5, mu5.max(), 60)
eff_sig = []
eff_mu  = []
for mt in mu_targets:
    cons = [cons_sum, {'type':'eq', 'fun': lambda w, mt=mt: w @ mu5 - mt}]
    res = minimize(port_var, x0=np.ones(n)/n, args=(Sigma5,), method='SLSQP',
                   bounds=bounds, constraints=cons)
    if res.success:
        eff_sig.append(np.sqrt(port_var(res.x, Sigma5)))
        eff_mu.append(mt)
eff_sig = np.array(eff_sig); eff_mu = np.array(eff_mu)

print('--- MVP (5 stocks) ---')
print(pd.Series(w_mvp5, index=names5, name='w_MVP').round(4))
print(f'E(R)={mu_mvp5:.4f}, σ={sig_mvp5:.4f}, Sharpe={(mu_mvp5-RF_ANN)/sig_mvp5:.4f}\n')

print('--- Market portfolio (5 stocks) ---')
print(pd.Series(w_mkt5, index=names5, name='w_MKT').round(4))
print(f'E(R)={mu_mkt5:.4f}, σ={sig_mkt5:.4f}, Sharpe={S_mkt5:.4f}')

In [ ]:
# --- 3. Plot: efficient frontier + MC cloud + comparison with Sec. 1 ---
fig, ax = plt.subplots(figsize=(12, 7))

sc = ax.scatter(mc_sig, mc_mu, c=mc_sharpe, cmap='viridis', s=8, alpha=0.5)
plt.colorbar(sc, ax=ax, label='Sharpe ratio')

ax.plot(eff_sig, eff_mu, color='red', lw=2.5, label='Efficient frontier (5 stocks)')
ax.plot(sig_p, mu_p, color='gray', lw=2, ls='--', label='Two-stock set (Sec. 1)')

# Individual stocks
for i, name in enumerate(names5):
    ax.scatter(np.sqrt(Sigma5[i,i]), mu5[i], s=110, marker='s',
               edgecolor='black', label=name, zorder=4)

ax.scatter(sig_mvp5, mu_mvp5, s=200, color='blue', marker='*',
           label=f'MVP (5)', zorder=5)
ax.scatter(sig_mkt5, mu_mkt5, s=200, color='red',  marker='*',
           label=f'Market portfolio (5)', zorder=5)
ax.scatter(sig_mvp, mu_mvp, s=140, color='lightblue', marker='*',
           edgecolor='navy', label='MVP (2)', zorder=4)
ax.scatter(sig_mkt, mu_mkt, s=140, color='lightcoral', marker='*',
           edgecolor='darkred', label='Market (2)', zorder=4)

x_cml5 = np.linspace(0, mc_sig.max()*1.05, 100)
ax.plot(x_cml5, RF_ANN + (mu_mkt5-RF_ANN)/sig_mkt5 * x_cml5, '--',
        color='red', alpha=0.5, label='CML (5)')

ax.axhline(RF_ANN, color='black', ls=':', alpha=0.5)
ax.set_xlabel('σ_p (annual)'); ax.set_ylabel('E(R_p) (annual)')
ax.set_title('Sec. 3: Efficient frontier for 5-stock portfolio (vs Sec. 1)',
             fontweight='bold')
ax.legend(loc='lower right', fontsize=9, ncol=2)
ax.grid(alpha=0.3); ax.set_xlim(left=0)
plt.tight_layout(); plt.show()

In [ ]:
# --- 3. Numerical comparison MVP/market: 2 stocks vs 5 stocks ---
rows = [
    ('MVP (2 stocks)',     mu_mvp,  sig_mvp,  (mu_mvp-RF_ANN)/sig_mvp),
    ('MVP (5 stocks)',     mu_mvp5, sig_mvp5, (mu_mvp5-RF_ANN)/sig_mvp5),
    ('Market (2 stocks)', mu_mkt,  sig_mkt,  (mu_mkt-RF_ANN)/sig_mkt),
    ('Market (5 stocks)', mu_mkt5, sig_mkt5, S_mkt5),
]
cmp_df = pd.DataFrame(rows, columns=['Portfolio','E(R)','σ','Sharpe']).round(4)
display(cmp_df)

# Capital allocation for market portfolio (5 stocks)
alloc = pd.DataFrame({
    'Weight': np.round(w_mkt5, 4),
    'Capital [PLN]': np.round(w_mkt5 * CAPITAL_PLN, 2),
}, index=names5)
print('\nAllocation of PLN 1,000,000 — market portfolio (5 stocks):')
display(alloc)

---
# 4. **Single-index** model (Sharpe single-index model)

Assumption: the return of stock $i$ decomposes into exposure to the market index $R_M$
and an idiosyncratic component:
$$
R_i = \alpha_i + \beta_i R_M + \varepsilon_i,\qquad
\mathrm{Cov}(\varepsilon_i, \varepsilon_j) = 0\ \text{for } i\neq j,\quad
\mathrm{Cov}(R_M, \varepsilon_i) = 0.
$$

Hence the covariance matrix **from the model**:
$$
\Sigma_{ij}^{(SIM)} = \beta_i \beta_j \sigma_M^2 + \mathbf{1}_{\{i=j\}}\sigma_{\varepsilon_i}^2.
$$

## 4a — Index: **WIG20**.

In [ ]:
# --- 4a/4b: estimation of alpha, beta, sigma2_eps for each stock ---
R_M = log_ret[INDEX_NAME].values                  # daily WIG20 log-returns
mu_M  = R_M.mean() * TRADING_DAYS
sig_M = R_M.std()  * np.sqrt(TRADING_DAYS)
var_M = sig_M ** 2

print(f'WIG20 index:  E(R_M) = {mu_M:.4f},  σ_M = {sig_M:.4f}')

sim_params = []
for name in names5:
    y = R5[name].values
    x = R_M
    # OLS: y = alpha + beta*x + eps
    cov_xy = np.cov(x, y, ddof=1)
    beta_d  = cov_xy[0, 1] / cov_xy[0, 0]
    alpha_d = y.mean() - beta_d * x.mean()
    resid   = y - (alpha_d + beta_d*x)
    sig2_eps_d = resid.var(ddof=2)
    # Annualization: alpha_year = alpha_day * 252; beta unchanged; var_eps_year = var_eps_day * 252
    sim_params.append({
        'Stock':   name,
        'alpha (yr)':    alpha_d * TRADING_DAYS,
        'beta':           beta_d,
        'sigma2_eps (yr)': sig2_eps_d * TRADING_DAYS,
        'R^2':     1 - resid.var(ddof=2) / y.var(ddof=1),
    })
sim_df = pd.DataFrame(sim_params).set_index('Stock').round(5)
display(sim_df)

alphas = sim_df['alpha (yr)'].values
betas  = sim_df['beta'].values
s2_eps = sim_df['sigma2_eps (yr)'].values

In [ ]:
# --- 4c: covariance matrix from single-index model ---
Sigma_SIM = np.outer(betas, betas) * var_M + np.diag(s2_eps)

print('Covariance matrix (single-index model, annual):')
display(pd.DataFrame(Sigma_SIM, index=names5, columns=names5).round(5))

print('\nCovariance matrix (sample, annual):')
display(pd.DataFrame(Sigma5, index=names5, columns=names5).round(5))

# Frobenius norm of the difference
diff_norm = np.linalg.norm(Sigma_SIM - Sigma5, 'fro')
print(f'\nFrobenius norm of difference ||Σ_SIM - Σ_sample||_F = {diff_norm:.5f}')

# Comparative plot of correlation matrices
def cov_to_corr(S):
    d = np.sqrt(np.diag(S))
    return S / np.outer(d, d)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, mat, title in zip(axes, [cov_to_corr(Sigma5), cov_to_corr(Sigma_SIM)],
                          ['Correlation — sample', 'Correlation — from single-index model']):
    im = ax.imshow(mat, cmap='RdBu_r', vmin=-1, vmax=1)
    ax.set_xticks(range(n)); ax.set_yticks(range(n))
    ax.set_xticklabels(names5, rotation=30, ha='right'); ax.set_yticklabels(names5)
    for i in range(n):
        for j in range(n):
            ax.text(j, i, f'{mat[i,j]:.2f}', ha='center', va='center',
                    color='black' if abs(mat[i,j])<0.5 else 'white', fontsize=9)
    ax.set_title(title, fontweight='bold')
    plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout(); plt.show()

## 4d — MVP and market portfolio **based on $\Sigma_{SIM}$**

We repeat the optimization from Section 3, but use the matrix from the single-index model.
We compute the expected return vector from the **SIM** formula: $\mathbb{E}(R_i) = \alpha_i + \beta_i \mathbb{E}(R_M)$
(one could also keep sample $\mu_5$ — we check both variants).

In [ ]:
# --- 4d: MVP and market portfolio from single-index model matrix ---
mu_SIM = alphas + betas * mu_M  # E(R_i) z SIM
# (alternatively: mu_SIM = mu5)

# MVP
res_mvp_S = minimize(port_var, x0=np.ones(n)/n, args=(Sigma_SIM,), method='SLSQP',
                     bounds=bounds, constraints=[cons_sum])
w_mvp_S = res_mvp_S.x
mu_mvp_S  = w_mvp_S @ mu_SIM
sig_mvp_S = np.sqrt(w_mvp_S @ Sigma_SIM @ w_mvp_S)

# Market portfolio
res_mkt_S = minimize(neg_sharpe, x0=np.ones(n)/n, args=(mu_SIM, Sigma_SIM, RF_ANN),
                     method='SLSQP', bounds=bounds, constraints=[cons_sum])
w_mkt_S = res_mkt_S.x
mu_mkt_S  = w_mkt_S @ mu_SIM
sig_mkt_S = np.sqrt(w_mkt_S @ Sigma_SIM @ w_mkt_S)
S_mkt_S   = (mu_mkt_S - RF_ANN) / sig_mkt_S

# Realized (ex post) risk of SIM portfolios with sample Σ
sig_mvp_S_realized = np.sqrt(w_mvp_S @ Sigma5 @ w_mvp_S)
sig_mkt_S_realized = np.sqrt(w_mkt_S @ Sigma5 @ w_mkt_S)

cmp_sim = pd.DataFrame([
    ['MVP — sample Σ', w_mvp5, mu_mvp5,  sig_mvp5,  sig_mvp5],
    ['MVP — SIM Σ',      w_mvp_S, mu_mvp_S, sig_mvp_S, sig_mvp_S_realized],
    ['Market — sample Σ', w_mkt5, mu_mkt5, sig_mkt5, sig_mkt5],
    ['Market — SIM Σ',      w_mkt_S, mu_mkt_S, sig_mkt_S, sig_mkt_S_realized],
], columns=['Portfolio','Weights','E(R)','σ (model)','σ realized (Σ_sample)'])
cmp_sim['E(R)']               = cmp_sim['E(R)'].round(4)
cmp_sim['σ (model)']          = cmp_sim['σ (model)'].round(4)
cmp_sim['σ realized (Σ_sample)']= cmp_sim['σ realized (Σ_sample)'].round(4)

print('MVP weights (SIM):',     pd.Series(w_mvp_S.round(4), index=names5).to_dict())
print('Market weights (SIM):', pd.Series(w_mkt_S.round(4), index=names5).to_dict())
display(cmp_sim[['Portfolio','E(R)','σ (model)','σ realized (Σ_sample)']])

In [ ]:
# --- Graphical comparison: SIM vs sample portfolios ---
fig, ax = plt.subplots(figsize=(11, 7))
ax.scatter(mc_sig, mc_mu, c='lightgray', s=6, alpha=0.4, label='MC (Σ sample)')
ax.plot(eff_sig, eff_mu, color='red', lw=2, label='Efficient frontier (Σ sample)')
for i, name in enumerate(names5):
    ax.scatter(np.sqrt(Sigma5[i,i]), mu5[i], s=80, marker='s', edgecolor='black')
    ax.annotate(name, (np.sqrt(Sigma5[i,i]), mu5[i]),
                textcoords='offset points', xytext=(5,5), fontsize=9)
ax.scatter(sig_mvp5, mu_mvp5, s=180, color='blue',  marker='*', label='MVP (Σ sample)')
ax.scatter(sig_mkt5, mu_mkt5, s=180, color='red',   marker='*', label='Market (Σ sample)')
ax.scatter(sig_mvp_S_realized, mu_mvp_S, s=180, color='cyan',   marker='X',
           label='MVP (Σ SIM, σ realized)', edgecolor='black')
ax.scatter(sig_mkt_S_realized, mu_mkt_S, s=180, color='orange', marker='X',
           label='Market (Σ SIM, σ realized)', edgecolor='black')
ax.axhline(RF_ANN, color='black', ls=':', alpha=0.5)
ax.set_xlabel('σ_p (annual)'); ax.set_ylabel('E(R_p) (annual)')
ax.set_title('Sec. 4d: Optimization on Σ_SIM vs sample Σ',
             fontweight='bold')
ax.legend(loc='lower right', fontsize=9); ax.grid(alpha=0.3); ax.set_xlim(left=0)
plt.tight_layout(); plt.show()

---
# 5. Interpretation of results

### Sec. 1 — two-asset portfolio (PKO BP + CD Projekt)
- Two stocks from different sectors (banks and tech) have noticeably different volatility (CD Projekt
  is typically significantly riskier). Correlation is **positive but low**, so
  the set of attainable portfolios clearly "bulges" to the left (diversification effect).
- **MVP** allocates a larger weight to the less volatile stock (PKO BP).
- The **market portfolio** (max Sharpe) usually lies between MVP and the highest-$E(R)$ portfolio
  — depending on r_f and the Sharpe ratios of individual stocks.

### Sec. 2 — VaR and 20% constraint
- The **variance-covariance** method assumes normally distributed returns — for individual stocks
  (especially CD Projekt) it understates "fat tails", but for 5-asset portfolios it is
  a reasonable approximation (central limit theorem effect at the portfolio level).
- The constraint "loss ≤ 20% per year with 99% prob." equals the condition
  $\mu_p - 2{,}3263\,\sigma_p \ge -0{,}20$. Portfolios close to MVP usually satisfy it,
  while 100% allocations to a single high-σ stock do not.
- The maximum $E(R)$ under the constraint is the **tangency point** of the efficient frontier with the line
  $\mu_p - 2{,}3263\,\sigma_p = -0{,}20$.

### Sec. 3 — diversification to 5 stocks
- Moving from 2 to 5 stocks, the **efficient frontier shifts left and up**: at the same
  risk one can achieve higher return, or at the same return — lower risk.
- Specifically: MVP(5) has lower σ than MVP(2), and the market portfolio(5) has a higher Sharpe than
  market(2) — sector diversification **works**, because we add assets with low
  pairwise correlations.

### Sec. 4 — single-index model
- Betas estimated relative to WIG20 place stocks in an intuitive order: KGHM and CD Projekt
  usually have β > 1 (cyclical, "aggressive"), Dino as a defensive player has β < 1, PKO BP
  has β ≈ 1.
- The $\Sigma_{SIM}$ matrix is **strongly regularized** — it enforces the same correlation structure
  (each pair of stocks correlates through one "common factor" = WIG20).
- In practice $\Sigma_{SIM}$ underestimates **sector correlations** (e.g. two stocks
  from the same sector correlated more strongly than through the common benchmark). Hence the
  Frobenius norm of the difference can be significant.
- Portfolios optimized on $\Sigma_{SIM}$ have **smoother weights** (less extreme
  allocations), but their realized risk (measured with sample $\Sigma$) is usually several to a dozen
  percent higher than MVP optimized directly on sample $\Sigma$.
- Advantage of SIM: **parameter reduction** from $\frac{n(n+1)}{2}$ to $2n+1$ → lower
  *overfitting* risk — especially with a short sample.

### Recommendation for XTB
- From a *risk-adjusted return* perspective, allocating PLN 1 million according to the **market
  portfolio (max Sharpe, 5 stocks)** is sensible — see allocation in Sec. 3.
- If management imposes a hard *risk budget* (VaR 99% ≤ 20%), we choose the **boundary
  portfolio from Sec. 2/3** designed for that constraint.
- The matrix from the **single-index model** should be treated as a *sanity check* — it shows
  how much portfolio risk comes from WIG20 exposure versus idiosyncratic risk.
